# Realty Income REIT - ML Models
## Step 8: Train and Register ML Models for Property Risk and Lease Renewal Prediction

This notebook trains two ML models:
1. **Lease Renewal Predictor** - Predicts probability of lease renewal based on property and tenant features
2. **Property Risk Scorer** - Classifies property risk level using financial and lease characteristics

In [ ]:
import snowflake.snowpark as snowpark
from snowflake.snowpark.context import get_active_session
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, mean_squared_error, classification_report
import warnings
warnings.filterwarnings('ignore')

session = get_active_session()

In [ ]:
%%sql -r lease_data
SELECT
    l.LEASE_ID,
    l.REMAINING_TERM_MONTHS,
    l.ANNUAL_RENT,
    l.RENT_ESCALATION_RATE,
    l.TRIPLE_NET,
    l.LEASE_TERM_MONTHS,
    t.CREDIT_RATING,
    t.ANNUAL_REVENUE,
    t.PUBLIC_COMPANY,
    p.PROPERTY_TYPE,
    p.CAP_RATE,
    p.SQUARE_FOOTAGE,
    p.OCCUPANCY_STATUS,
    CASE WHEN l.STATUS = 'Active' AND l.REMAINING_TERM_MONTHS > 0 THEN 1 ELSE 0 END AS RENEWED
FROM REALTY_INCOME.RAW.LEASES l
JOIN REALTY_INCOME.RAW.TENANTS t ON l.TENANT_ID = t.TENANT_ID
JOIN REALTY_INCOME.RAW.PROPERTIES p ON l.PROPERTY_ID = p.PROPERTY_ID

In [ ]:
df = lease_data.copy()

le_credit = LabelEncoder()
le_property = LabelEncoder()
le_occupancy = LabelEncoder()

df['CREDIT_RATING_ENC'] = le_credit.fit_transform(df['CREDIT_RATING'].fillna('NR'))
df['PROPERTY_TYPE_ENC'] = le_property.fit_transform(df['PROPERTY_TYPE'].fillna('Unknown'))
df['OCCUPANCY_ENC'] = le_occupancy.fit_transform(df['OCCUPANCY_STATUS'].fillna('Unknown'))
df['TRIPLE_NET_INT'] = df['TRIPLE_NET'].astype(int)
df['PUBLIC_COMPANY_INT'] = df['PUBLIC_COMPANY'].astype(int)

features = ['REMAINING_TERM_MONTHS', 'ANNUAL_RENT', 'RENT_ESCALATION_RATE',
            'TRIPLE_NET_INT', 'LEASE_TERM_MONTHS', 'CREDIT_RATING_ENC',
            'ANNUAL_REVENUE', 'PUBLIC_COMPANY_INT', 'PROPERTY_TYPE_ENC',
            'CAP_RATE', 'SQUARE_FOOTAGE', 'OCCUPANCY_ENC']

X = df[features].fillna(0)
y = df['RENEWED']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Training samples: {len(X_train)}')
print(f'Test samples: {len(X_test)}')
print(f'Renewal rate: {y.mean():.2%}')

In [ ]:
renewal_model = GradientBoostingClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)
renewal_model.fit(X_train, y_train)

y_pred = renewal_model.predict(X_test)
y_prob = renewal_model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
print(f'Lease Renewal Model Accuracy: {accuracy:.4f}')
print(f'\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Not Renewed', 'Renewed']))

feature_importance = pd.DataFrame({
    'feature': features,
    'importance': renewal_model.feature_importances_
}).sort_values('importance', ascending=False)
print('\nTop Feature Importances:')
print(feature_importance.to_string(index=False))

In [ ]:
%%sql -r risk_data
SELECT
    r.RISK_ID,
    r.RISK_SCORE,
    r.RISK_LEVEL,
    r.GEOGRAPHIC_RISK,
    r.TENANT_CREDIT_RISK,
    r.MARKET_RISK,
    r.INTEREST_RATE_RISK,
    r.ENVIRONMENTAL_RISK,
    r.LEASE_EXPIRATION_RISK,
    p.PROPERTY_TYPE,
    p.CAP_RATE,
    p.SQUARE_FOOTAGE,
    p.YEAR_BUILT,
    l.REMAINING_TERM_MONTHS,
    l.ANNUAL_RENT,
    t.CREDIT_RATING
FROM REALTY_INCOME.RAW.RISK_ASSESSMENTS r
JOIN REALTY_INCOME.RAW.PROPERTIES p ON r.PROPERTY_ID = p.PROPERTY_ID
LEFT JOIN REALTY_INCOME.RAW.LEASES l ON p.PROPERTY_ID = l.PROPERTY_ID
LEFT JOIN REALTY_INCOME.RAW.TENANTS t ON l.TENANT_ID = t.TENANT_ID

In [ ]:
df_risk = risk_data.copy()

le_risk_prop = LabelEncoder()
le_risk_credit = LabelEncoder()
le_risk_level = LabelEncoder()

df_risk['PROPERTY_TYPE_ENC'] = le_risk_prop.fit_transform(df_risk['PROPERTY_TYPE'].fillna('Unknown'))
df_risk['CREDIT_RATING_ENC'] = le_risk_credit.fit_transform(df_risk['CREDIT_RATING'].fillna('NR'))
df_risk['RISK_LEVEL_ENC'] = le_risk_level.fit_transform(df_risk['RISK_LEVEL'].fillna('Medium'))

risk_features = ['GEOGRAPHIC_RISK', 'TENANT_CREDIT_RISK', 'MARKET_RISK',
                 'INTEREST_RATE_RISK', 'ENVIRONMENTAL_RISK', 'LEASE_EXPIRATION_RISK',
                 'CAP_RATE', 'SQUARE_FOOTAGE', 'YEAR_BUILT',
                 'REMAINING_TERM_MONTHS', 'ANNUAL_RENT', 'PROPERTY_TYPE_ENC', 'CREDIT_RATING_ENC']

X_risk = df_risk[risk_features].fillna(0)
y_risk = df_risk['RISK_LEVEL_ENC']

X_risk_train, X_risk_test, y_risk_train, y_risk_test = train_test_split(
    X_risk, y_risk, test_size=0.2, random_state=42
)

risk_model = GradientBoostingClassifier(
    n_estimators=150,
    max_depth=6,
    learning_rate=0.1,
    random_state=42
)
risk_model.fit(X_risk_train, y_risk_train)

y_risk_pred = risk_model.predict(X_risk_test)
risk_accuracy = accuracy_score(y_risk_test, y_risk_pred)

print(f'Property Risk Model Accuracy: {risk_accuracy:.4f}')
print(f'\nClassification Report:')
risk_labels = le_risk_level.classes_
print(classification_report(y_risk_test, y_risk_pred, target_names=risk_labels))

risk_feature_importance = pd.DataFrame({
    'feature': risk_features,
    'importance': risk_model.feature_importances_
}).sort_values('importance', ascending=False)
print('\nTop Feature Importances:')
print(risk_feature_importance.to_string(index=False))

In [ ]:
from snowflake.ml.registry import Registry

reg = Registry(session=session, database_name='REALTY_INCOME', schema_name='ML')

sample_input_renewal = X_test.head(10)
renewal_mv = reg.log_model(
    model=renewal_model,
    model_name='LEASE_RENEWAL_PREDICTOR',
    version_name='v1',
    sample_input_data=sample_input_renewal,
    comment='Gradient Boosting model predicting lease renewal probability based on tenant credit, lease terms, and property characteristics'
)
print(f'Lease Renewal model registered: REALTY_INCOME.ML.LEASE_RENEWAL_PREDICTOR v1')

sample_input_risk = X_risk_test.head(10)
risk_mv = reg.log_model(
    model=risk_model,
    model_name='PROPERTY_RISK_CLASSIFIER',
    version_name='v1',
    sample_input_data=sample_input_risk,
    comment='Gradient Boosting classifier predicting property risk level (Low/Medium/High) from multi-factor risk inputs'
)
print(f'Property Risk model registered: REALTY_INCOME.ML.PROPERTY_RISK_CLASSIFIER v1')

In [ ]:
print('='*60)
print('ML MODEL TRAINING SUMMARY')
print('='*60)
print(f'\n1. Lease Renewal Predictor')
print(f'   - Algorithm: GradientBoostingClassifier')
print(f'   - Accuracy: {accuracy:.4f}')
print(f'   - Top Feature: {feature_importance.iloc[0]["feature"]}')
print(f'   - Registry: REALTY_INCOME.ML.LEASE_RENEWAL_PREDICTOR v1')
print(f'\n2. Property Risk Classifier')
print(f'   - Algorithm: GradientBoostingClassifier')
print(f'   - Accuracy: {risk_accuracy:.4f}')
print(f'   - Top Feature: {risk_feature_importance.iloc[0]["feature"]}')
print(f'   - Registry: REALTY_INCOME.ML.PROPERTY_RISK_CLASSIFIER v1')
print(f'\n' + '='*60)